# Getting started with the WCC ETC

The Wide-field Context Camera (WCC) Exposure Time Calculator estimates the
signal-to-noise ratio (SNR) a source reaches in a given exposure. The workflow is
always the same three steps:

1. **Build a scene** — the astrophysical source plus its background (`get_scene`).
2. **Build a simulation** for a sensor + filter (`Simulation.from_sensor_and_scene`).
3. **Ask it questions** — SNR for a time, time for an SNR, peak pixel, saturation.

This notebook walks through that loop. The other notebooks go deeper on individual
features — see **Where to go next** at the end.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import wcc_etc

wcc_etc.set_wcc_style()

## 1. Build a scene

`get_scene` returns what you want to observe together with its background. Here a
Sun-like **G5V** star at **r = 20 mag** on a **zodiacal** background. The `name` can be a
spectral type (`'G5V'`, `'K3IV'`, ...) or a parametric spectrum (`'blackbody'`, `'flat'`,
`'powerlaw'`, `'emission'`) — see `06_source_spectra.ipynb`.

In [ ]:
scene = wcc_etc.get_scene(
    name="G5V",
    mag=20,
    background="zodi",
    bandpass="johnson_r",
    background_prop={"bandpass": "johnson_r", "mag": 22.5},
)
scene

## 2. Build a simulation for a sensor + filter

`Simulation.from_sensor_and_scene` pairs the scene with a detector + filter, given as a
`kind:band` label (`'sony:r'`, `'sony:bb'`, `'qcmos:r'`). Pixel size, gain, read noise,
dark current, and filter throughput all come from the bundled config files.

In [ ]:
sim = wcc_etc.Simulation.from_sensor_and_scene("sony:r", scene)

print("pixel size :", sim.sensor.pixel_size)
print("bit depth  :", sim.sensor.bit_depth, "bit")
print("read noise :", sim.sensor.read_noise)

## 3. Compute the SNR

`get_snr(time)` runs the 2-D image simulation and returns a dict; `["snr"]` is the
signal-to-noise. The time can be a scalar or an array — it broadcasts — so a single call
gives an SNR-vs-time curve.

In [ ]:
times = np.linspace(1, 300, 100)
snr = sim.get_snr(time=times)["snr"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(times, snr)
ax.axvline(60, ls="--", color="0.6")
ax.set_xlabel("Exposure time [s]")
ax.set_ylabel("SNR")
ax.set_title("G5V, r = 20, sony:r")
plt.show()

## 4. Inspect and change parameters

`sim.mutable_parameters` lists every tunable parameter and `sim.meta` holds the current
values. `sim.update(**kwargs)` changes them in place — a double underscore reaches into a
nested field, e.g. `source__mag`.

Two magnitudes of extra brightness take the 60 s SNR from **120.2 to 306.7** — a
factor of **2.55**, close to the √6.31 = 2.51 you expect from Pogson scaling when the
measurement is background limited.

Jitter is the more interesting one. Switching the telescope's default 10 mas off barely
moves the SNR (**306.7 → 307.3**, 0.2%), because jitter redistributes light *inside* the
aperture rather than out of it. What it does move is the **peak-pixel fraction**, from
**0.0956 to 0.1228** — 28% more light in the brightest pixel, which is a saturation
question rather than an SNR one. `02_saturation_flag.ipynb` picks that up.

In [ ]:
print("mutable parameters:", sim.mutable_parameters[:8], "...")

before = float(sim.get_snr(60)["snr"])
sim.update(source__mag=18)  # brighter source
after = float(sim.get_snr(60)["snr"])

print(f"SNR @ 60 s, r = 20 : {before:.1f}")
print(f"SNR @ 60 s, r = 18 : {after:.1f}")

# A double underscore reaches a nested field. The telescope carries 10 mas of
# jitter by default, so switching it off is an update to the telescope, not the
# simulation -- `jitter_sigma=` alone would not match anything.
sim.update(telescope__jitter_sigma=0)
print(f"SNR @ 60 s, r = 18, jitter off : {float(sim.get_snr(60)['snr']):.1f}")
print(f"peak-pixel fraction, jitter off : {sim.peak_pixel_fraction():.4f}")
sim.update(telescope__jitter_sigma=10)
print(f"peak-pixel fraction, 10 mas     : {sim.peak_pixel_fraction():.4f}")

## 5. Check what you actually set up

The scene source and the sensor both know how to plot themselves, which is the quickest
sanity check that you built the source and filter you intended.

In [ ]:
fig, (ax_src, ax_filt) = plt.subplots(1, 2, figsize=(11, 4))

sim.scene.source.show(ax=ax_src, wave=np.arange(3500, 9500, 5.0), flux_unit="flam")
ax_src.set_title("Source spectrum (G5V, r = 18)")

sim.sensor.show(ax=ax_filt)
ax_filt.set_title("Filter throughput (sony:r)")

fig.tight_layout()
plt.show()

## Where to go next

| Notebook | What it adds |
|---|---|
| `02_saturation_flag.ipynb` | Peak-pixel value and saturation flagging. |
| `03_from_sensorfilter.ipynb` | Canonical `kind:band` labels with the PSF picked from the filter's focus level. |
| `04_psf_and_image_snr.ipynb` | PSF models, image simulation, and PSF-aware aperture SNR. |
| `05_n_reads_exptime.ipynb` | Multiple reads (`n_reads`) and the exposure-time inverses. |
| `06_source_spectra.ipynb` | Parametric source spectra and normalization bandpasses. |

## Summary

- **Scene → Simulation → question** is the whole workflow.
- `get_scene(name=..., mag=..., background=...)` builds the source + background.
- `Simulation.from_sensor_and_scene('sony:r', scene)` pairs it with a detector + filter.
- `sim.get_snr(time)` takes a scalar or an array of times.
- `sim.mutable_parameters` lists what you can change; `sim.update(source__mag=18)` changes it.
- `scene.source.show()` and `sensor.show()` plot the spectrum and the filter curve.